# 01 — Download 10-K filings from SEC EDGAR

**Goal:** for each ticker, download the most recent **10-K filed strictly before April 2, 2025** (the tariff announcement date).

**Why the date rule matters:** if we used a 10-K filed *after* the announcement, management could be describing the tariffs after the fact. That is look-ahead bias, and the auditor will check it.

| | |
|---|---|
| **Inputs** | SEC EDGAR APIs (free, no key; requires a contact email) |
| **Outputs** | `data/raw/edgar_tariffs/{TICKER}_10K_{date}.html` (git-ignored, recreated by this notebook)<br>`data/processed/tenk_manifest.csv` (one row per firm: which filing we used, its URL, SIC code, shares outstanding)<br>`data/processed/download_problems.csv` (every ticker we could not get, and why) |


In [1]:
# --- Setup: find the repo root and load shared settings from code/config.py ---
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "run_all.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Can't find the repo root. Run this from inside the project folder.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "code"))

import config as cfg
print("Repo root:", ROOT)

Repo root: /content/project


In [2]:
import time
from datetime import datetime, timezone
import pandas as pd
import requests

if "@" not in cfg.SEC_EMAIL:
    raise ValueError(
        "SEC requires a contact email. In Colab, run this first:  %env SEC_EMAIL=yourname@lehigh.edu"
    )

cfg.EDGAR_DIR.mkdir(parents=True, exist_ok=True)
cfg.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def sec_get(url):
    """GET a URL from SEC politely: pause between calls (SEC limit is 10 per second)."""
    time.sleep(0.15)
    r = requests.get(url, headers=cfg.SEC_HEADERS, timeout=60)
    r.raise_for_status()
    return r

## Step 1: map tickers to SEC CIK numbers
SEC identifies firms by a 10-digit **CIK**, not by ticker. SEC publishes the ticker→CIK list.

*Look-ahead trap:* this list is **current**, not as of 2025. A ticker can now point to a different, newer registrant than the one that filed in early 2025. That happened with XOM: today's list maps it to a new CIK with no pre-2025 10-K. `CIK_OVERRIDES` in `config.py` pins the historical CIK, and the `cik_source` column records where each CIK came from. Any firm that still fails is logged in `download_problems.csv`, not silently dropped. For the full sample we'll take CIKs from a point-in-time S&P 500 list instead.

In [3]:
tickers_json = sec_get("https://www.sec.gov/files/company_tickers.json").json()
ticker_to_cik = {row["ticker"].upper(): str(row["cik_str"]).zfill(10) for row in tickers_json.values()}
print(f"SEC ticker list has {len(ticker_to_cik):,} tickers")
{t: ticker_to_cik.get(t) for t in cfg.TICKERS}

SEC ticker list has 10,404 tickers


{'AAPL': '0000320193',
 'NKE': '0000320187',
 'BBY': '0000764478',
 'TGT': '0000027419',
 'COST': '0000909832',
 'CAT': '0000018230',
 'TSLA': '0001318605',
 'MSFT': '0000789019',
 'JNJ': '0000200406',
 'XOM': '0002115436'}

## Step 2: helper functions

- `find_last_10k_before_cutoff`: looks through the firm's filing history. Big filers (e.g., banks) file so often that the "recent" list may not reach back to early 2025, so we also check the older history pages.
- `shares_from_10k`: shares outstanding as reported **in that same 10-K** (from SEC's XBRL data). This gives us a pre-event market cap with no look-ahead. We use the cover-page count first. If a firm has several share classes (e.g., Nike Class A and B), the cover count is often missing in SEC's summary data, so we fall back to weighted-average basic shares from the same 10-K. The `shares_source` column records which one we used.

In [4]:
def filings_table(block):
    """Turn SEC's column-oriented filing lists into a DataFrame."""
    cols = ["form", "filingDate", "reportDate", "accessionNumber", "primaryDocument"]
    return pd.DataFrame({c: block[c] for c in cols})


def find_last_10k_before_cutoff(cik):
    sub = sec_get(f"https://data.sec.gov/submissions/CIK{cik}.json").json()
    tables = [filings_table(sub["filings"]["recent"])]
    for page in sub["filings"].get("files", []):          # older filing history, if any
        tables.append(filings_table(sec_get(f"https://data.sec.gov/submissions/{page['name']}").json()))
    filings = pd.concat(tables, ignore_index=True)

    # Exact form "10-K" only (excludes amendments "10-K/A"); filed strictly before the cutoff
    tenks = filings[(filings["form"] == "10-K") & (filings["filingDate"] < cfg.TENK_CUTOFF)]
    tenks = tenks.sort_values("filingDate", ascending=False)
    return sub, (tenks.iloc[0] if len(tenks) else None)


def shares_from_10k(cik, accession):
    """Shares outstanding reported in the given 10-K. Returns (shares, source) or (None, reason)."""
    facts = sec_get(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json").json().get("facts", {})

    # 1) Cover page: dei:EntityCommonStockSharesOutstanding
    cover = facts.get("dei", {}).get("EntityCommonStockSharesOutstanding", {}).get("units", {}).get("shares", [])
    cover = [e for e in cover if e.get("accn") == accession]
    if cover:
        last = max(e["end"] for e in cover)
        return sum(e["val"] for e in cover if e["end"] == last), "cover_page"

    # 2) Fallback: weighted-average basic shares for the fiscal year, same filing
    wavg = (facts.get("us-gaap", {}).get("WeightedAverageNumberOfSharesOutstandingBasic", {})
                 .get("units", {}).get("shares", []))
    wavg = [e for e in wavg if e.get("accn") == accession and e.get("start")
            and (pd.Timestamp(e["end"]) - pd.Timestamp(e["start"])).days > 300]
    if wavg:
        last = max(wavg, key=lambda e: e["end"])
        return last["val"], "weighted_avg_basic"
    return None, "not_found"

## Step 3: download each firm's filing

In [5]:
rows, problems = [], []

for t in cfg.TICKERS:
    if t in cfg.CIK_OVERRIDES:
        cik, cik_source = cfg.CIK_OVERRIDES[t], "config override (point-in-time)"
    else:
        cik, cik_source = ticker_to_cik.get(t), "SEC current ticker list"
    if cik is None:
        problems.append({"ticker": t, "step": "01_edgar", "problem": "ticker not in SEC list"})
        continue
    try:
        sub, f = find_last_10k_before_cutoff(cik)
        if f is None:
            problems.append({"ticker": t, "step": "01_edgar",
                             "problem": f"no 10-K filed before {cfg.TENK_CUTOFF} under CIK {cik} ({cik_source}); "
                                        "if this firm existed then, its historical CIK may differ: add it to CIK_OVERRIDES"})
            continue

        url = (f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/"
               f"{f['accessionNumber'].replace('-', '')}/{f['primaryDocument']}")
        local = cfg.EDGAR_DIR / f"{t}_10K_{f['filingDate']}.html"
        if not local.exists():                      # skip re-downloading on reruns
            local.write_bytes(sec_get(url).content)

        shares, shares_source = shares_from_10k(cik, f["accessionNumber"])

        rows.append({
            "ticker": t,
            "cik": cik,
            "cik_source": cik_source,
            "company": sub.get("name"),
            "sic": sub.get("sic"),
            "sic_description": sub.get("sicDescription"),
            "filing_date": f["filingDate"],
            "fiscal_year_end": f["reportDate"],
            "accession": f["accessionNumber"],
            "url": url,
            "local_file": local.name,
            "shares_outstanding": shares,
            "shares_source": shares_source,
            "downloaded_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
        })
        print(f"[EDGAR] {t}: 10-K filed {f['filingDate']} (FY end {f['reportDate']}), shares: {shares_source}")
    except Exception as e:
        problems.append({"ticker": t, "step": "01_edgar", "problem": f"error: {e}"})
        print(f"[EDGAR] {t}: FAILED - {e}")

manifest = pd.DataFrame(rows)
manifest.to_csv(cfg.MANIFEST_CSV, index=False)
pd.DataFrame(problems, columns=["ticker", "step", "problem"]).to_csv(cfg.DOWNLOAD_PROBLEMS_CSV, index=False)
print(f"\nSaved {len(manifest)} filings to manifest; {len(problems)} problems logged.")

[EDGAR] AAPL: 10-K filed 2024-11-01 (FY end 2024-09-28), shares: cover_page


[EDGAR] NKE: 10-K filed 2024-07-25 (FY end 2024-05-31), shares: weighted_avg_basic


[EDGAR] BBY: 10-K filed 2025-03-19 (FY end 2025-02-01), shares: cover_page


[EDGAR] TGT: 10-K filed 2025-03-12 (FY end 2025-02-01), shares: cover_page


[EDGAR] COST: 10-K filed 2024-10-09 (FY end 2024-09-01), shares: cover_page


[EDGAR] CAT: 10-K filed 2025-02-14 (FY end 2024-12-31), shares: cover_page


[EDGAR] TSLA: 10-K filed 2025-01-30 (FY end 2024-12-31), shares: cover_page


[EDGAR] MSFT: 10-K filed 2024-07-30 (FY end 2024-06-30), shares: cover_page


[EDGAR] JNJ: 10-K filed 2025-02-13 (FY end 2024-12-29), shares: cover_page


[EDGAR] XOM: 10-K filed 2025-02-19 (FY end 2024-12-31), shares: cover_page

Saved 10 filings to manifest; 0 problems logged.


## Checks (these stop the pipeline if something is wrong)

In [6]:
assert manifest["ticker"].is_unique, "duplicate tickers in manifest"
assert (manifest["filing_date"] < cfg.TENK_CUTOFF).all(), "LOOK-AHEAD: a 10-K was filed on/after the cutoff"
assert all((cfg.EDGAR_DIR / f).exists() for f in manifest["local_file"]), "a downloaded file is missing"
manifest[["ticker", "company", "filing_date", "fiscal_year_end", "sic", "shares_outstanding", "shares_source"]]

,ticker,company,filing_date,fiscal_year_end,sic,shares_outstanding,shares_source
0,AAPL,Apple Inc.,2024-11-01,2024-09-28,3571,15115823000,cover_page
1,NKE,"NIKE, Inc.",2024-07-25,2024-05-31,3021,1517600000,weighted_avg_basic
2,BBY,BEST BUY CO INC,2025-03-19,2025-02-01,5731,211369657,cover_page
3,TGT,TARGET CORP,2025-03-12,2025-02-01,5331,455576464,cover_page
4,COST,COSTCO WHOLESALE CORP /NEW,2024-10-09,2024-09-01,5331,443073537,cover_page
5,CAT,CATERPILLAR INC,2025-02-14,2024-12-31,3531,477932024,cover_page
6,TSLA,"Tesla, Inc.",2025-01-30,2024-12-31,3711,3216517037,cover_page
7,MSFT,MICROSOFT CORP,2024-07-30,2024-06-30,7372,7433038381,cover_page
8,JNJ,JOHNSON & JOHNSON,2025-02-13,2024-12-29,2834,2407616693,cover_page
9,XOM,EXXON MOBIL CORP,2025-02-19,2024-12-31,2911,4339143313,cover_page
